# 2026 World Cup Results vs Model Predictions

This notebook audits every 2026 FIFA World Cup match currently present in the local processed dataset. For each match date, it trains the best backtested market-free model from the model sweep, weighted L2 logistic regression, using only matches before that date, then compares the prediction with the actual result.

The generated CSV and README table are based on local data through 2026-07-03.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed' / 'matches_with_elo.csv'
REPORTS = ROOT / 'reports'
SEED = 20260707
LABELS = {'H': 0, 'D': 1, 'A': 2}
ORDERED = np.array(['H', 'D', 'A'])

matches = pd.read_csv(DATA, parse_dates=['date']).sort_values(['date', 'match_id']).reset_index(drop=True)
wc2026 = matches[(matches.tournament == 'FIFA World Cup') & (matches.date.dt.year == 2026)].copy()
assert len(wc2026) == 88
assert wc2026.date.max() == pd.Timestamp('2026-07-03')
wc2026[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'result']].head()


,date,home_team,away_team,home_score,away_score,result
49405,2026-06-11,Mexico,South Africa,2,0,H
49406,2026-06-11,South Korea,Czech Republic,2,1,H
49407,2026-06-12,Canada,Bosnia and Herzegovina,1,1,D
49408,2026-06-12,United States,Paraguay,4,1,H
49409,2026-06-13,Qatar,Switzerland,1,1,D


In [2]:
def classify_tournament(name):
    text = str(name).lower()
    if 'qualif' in text:
        return 'qualifier'
    if str(name) == 'FIFA World Cup':
        return 'world_cup'
    markers = ['copa america', 'copa américa', 'african cup of nations', 'afc asian cup', 'uefa euro', 'european championship', 'gold cup', 'oceania', 'nations league']
    if any(marker in text for marker in markers):
        return 'continental'
    if text == 'friendly':
        return 'friendly'
    return 'other'

def add_world_cup_stage(frame):
    frame = frame.copy()
    frame['wc_match_number'] = -1
    mask = frame.tournament.eq('FIFA World Cup')
    frame.loc[mask, 'wc_match_number'] = frame.loc[mask].groupby(frame.loc[mask, 'date'].dt.year).cumcount()
    frame['wc_group_stage'] = frame.wc_match_number.between(0, 47).astype(float)
    frame['wc_knockout_stage'] = (frame.wc_match_number >= 48).astype(float)
    return frame

def build_team_panel(frame):
    points = {'H': 1.0, 'D': 0.5, 'A': 0.0}
    home = pd.DataFrame({
        'match_id': frame.match_id, 'date': frame.date, 'team': frame.home_team, 'is_home': True,
        'goals_for': frame.home_score.astype(float), 'goals_against': frame.away_score.astype(float),
        'team_elo': frame.home_elo.astype(float), 'result_points': frame.result.map(points).astype(float),
        'expected_points': frame.elo_home_probability + 0.5 * frame.elo_draw_probability,
    })
    away = pd.DataFrame({
        'match_id': frame.match_id, 'date': frame.date, 'team': frame.away_team, 'is_home': False,
        'goals_for': frame.away_score.astype(float), 'goals_against': frame.home_score.astype(float),
        'team_elo': frame.away_elo.astype(float), 'result_points': 1.0 - frame.result.map(points).astype(float),
        'expected_points': frame.elo_away_probability + 0.5 * frame.elo_draw_probability,
    })
    panel = pd.concat([home, away], ignore_index=True).sort_values(['team', 'date', 'match_id'], kind='mergesort').reset_index(drop=True)
    grouped = panel.groupby('team', sort=False)
    panel['days_since_match'] = grouped.date.diff().dt.days.fillna(365).clip(0, 365)
    recent_counts = pd.Series(0.0, index=panel.index)
    for _, idx in panel.groupby('team', sort=False).groups.items():
        dates = panel.loc[idx, 'date'].to_numpy()
        counts = []
        for pos, current in enumerate(dates):
            prior = dates[:pos]
            counts.append(float(np.sum((current - prior) <= np.timedelta64(365, 'D'))))
        recent_counts.loc[idx] = counts
    panel['recent_matches_365'] = recent_counts
    for window in [3, 5, 10]:
        for col in ['goals_for', 'goals_against', 'result_points']:
            fill = {'goals_for': 1.25, 'goals_against': 1.25, 'result_points': 0.5}[col]
            panel[f'rolling_{window}_{col}'] = grouped[col].transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean()).fillna(fill)
        actual = grouped.result_points.transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean())
        expected = grouped.expected_points.transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean())
        panel[f'rolling_{window}_form_vs_expected'] = (actual - expected).fillna(0.0)
        panel[f'rolling_{window}_elo_trend'] = (grouped.team_elo.shift(1) - grouped.team_elo.shift(1 + window)).fillna(0.0)
    return panel

def build_features(frame):
    frame = add_world_cup_stage(frame)
    frame['tournament_type'] = frame.tournament.map(classify_tournament)
    panel = build_team_panel(frame)
    value_cols = [c for c in panel.columns if c.startswith('rolling_')] + ['days_since_match', 'recent_matches_365']
    home = panel.loc[panel.is_home, ['match_id', *value_cols]].rename(columns={c: f'home_{c}' for c in value_cols})
    away = panel.loc[~panel.is_home, ['match_id', *value_cols]].rename(columns={c: f'away_{c}' for c in value_cols})
    out = frame.merge(home, on='match_id', how='left').merge(away, on='match_id', how='left')
    out['abs_elo_difference'] = out.elo_difference.abs()
    out['elo_similarity'] = np.exp(-out.abs_elo_difference / 200.0)
    out['elo_difference_sq'] = np.sign(out.elo_difference) * (out.elo_difference / 400.0) ** 2
    for window in [3, 5, 10]:
        out[f'recent_goal_total_{window}'] = out[f'home_rolling_{window}_goals_for'] + out[f'away_rolling_{window}_goals_for']
        out[f'recent_defence_total_{window}'] = out[f'home_rolling_{window}_goals_against'] + out[f'away_rolling_{window}_goals_against']
        out[f'recent_form_gap_{window}'] = out[f'home_rolling_{window}_result_points'] - out[f'away_rolling_{window}_result_points']
        out[f'recent_attack_gap_{window}'] = out[f'home_rolling_{window}_goals_for'] - out[f'away_rolling_{window}_goals_for']
        out[f'recent_defence_gap_{window}'] = out[f'home_rolling_{window}_goals_against'] - out[f'away_rolling_{window}_goals_against']
    out['rest_gap'] = out.home_days_since_match - out.away_days_since_match
    out['activity_gap_365'] = out.home_recent_matches_365 - out.away_recent_matches_365
    return out.sort_values(['date', 'match_id'], kind='mergesort').reset_index(drop=True)

featured = build_features(matches)
assert featured.filter(regex='rolling_|days_since|recent_|elo_similarity|stage|gap').isna().sum().sum() == 0
featured.shape


(49493, 80)

In [3]:
FEATURES = [
    'elo_difference', 'abs_elo_difference', 'elo_difference_sq', 'elo_similarity', 'neutral', 'home_elo', 'away_elo', 'wc_group_stage', 'wc_knockout_stage',
    'home_days_since_match', 'away_days_since_match', 'home_recent_matches_365', 'away_recent_matches_365', 'rest_gap', 'activity_gap_365',
]
for window in [3, 5, 10]:
    FEATURES += [
        f'home_rolling_{window}_goals_for', f'home_rolling_{window}_goals_against', f'home_rolling_{window}_result_points', f'home_rolling_{window}_form_vs_expected', f'home_rolling_{window}_elo_trend',
        f'away_rolling_{window}_goals_for', f'away_rolling_{window}_goals_against', f'away_rolling_{window}_result_points', f'away_rolling_{window}_form_vs_expected', f'away_rolling_{window}_elo_trend',
        f'recent_goal_total_{window}', f'recent_defence_total_{window}', f'recent_form_gap_{window}', f'recent_attack_gap_{window}', f'recent_defence_gap_{window}',
    ]
TOURNAMENT_LEVELS = ['world_cup', 'qualifier', 'continental', 'friendly']

def design(frame):
    out = frame[FEATURES].astype(float).copy()
    for col in [c for c in out.columns if 'elo' in c or c in ['home_elo', 'away_elo', 'elo_difference', 'abs_elo_difference']]:
        out[col] = out[col] / 400.0
    for col in ['home_days_since_match', 'away_days_since_match', 'rest_gap']:
        out[col] = out[col] / 30.0
    out['neutral'] = frame.neutral.astype(float)
    for level in TOURNAMENT_LEVELS:
        out[f'tournament_{level}'] = (frame.tournament_type == level).astype(float)
    return out

def labels_of(frame):
    return frame.result.map(LABELS).to_numpy()

def sample_weights(frame):
    age_years = (frame.date.max() - frame.date).dt.days / 365.25
    recency = np.exp(-age_years / 7.0)
    type_weight = frame.tournament_type.map({'world_cup': 1.7, 'qualifier': 1.25, 'continental': 1.15, 'friendly': 0.65, 'other': 1.0}).fillna(1.0)
    return np.asarray(recency * type_weight, dtype=float)

def normalize(probs):
    probs = np.clip(np.asarray(probs, dtype=float), 1e-12, None)
    return probs / probs.sum(axis=1, keepdims=True)

rows = []
for match_date in sorted(wc2026.date.unique()):
    match_date = pd.Timestamp(match_date)
    train = featured[(featured.date < match_date) & (featured.date >= '2000-01-01')].copy()
    day = featured[(featured.tournament == 'FIFA World Cup') & (featured.date.dt.year == 2026) & (featured.date == match_date)].copy()
    assert len(train) > 0 and train.date.max() < match_date and len(day) > 0
    model = make_pipeline(StandardScaler(), LogisticRegression(C=0.25, max_iter=3000, random_state=SEED))
    model.fit(design(train), labels_of(train), logisticregression__sample_weight=sample_weights(train))
    probs = normalize(model.predict_proba(design(day)))
    for idx, row in enumerate(day.itertuples(index=False)):
        pred_code = ORDERED[int(probs[idx].argmax())]
        actual_code = row.result
        pred_label = row.home_team if pred_code == 'H' else row.away_team if pred_code == 'A' else 'Draw'
        actual_label = row.home_team if actual_code == 'H' else row.away_team if actual_code == 'A' else 'Draw'
        rows.append({
            'date': row.date.date().isoformat(),
            'match': f'{row.home_team} vs {row.away_team}',
            'score': f'{int(row.home_score)}-{int(row.away_score)}',
            'model_prediction': pred_label,
            'actual_result': actual_label,
            'correct': 'Yes' if pred_code == actual_code else 'No',
            'home_win_probability': probs[idx, 0],
            'draw_probability': probs[idx, 1],
            'away_win_probability': probs[idx, 2],
        })

audit = pd.DataFrame(rows)
accuracy = audit.correct.eq('Yes').mean()
REPORTS.mkdir(exist_ok=True)
audit.to_csv(REPORTS / 'world_cup_2026_results_vs_predictions.csv', index=False)

readme_table = audit[['date', 'match', 'score', 'model_prediction', 'actual_result', 'correct']].copy()
readme_table.columns = ['Date', 'Match', 'Score', 'Model pick', 'Actual', 'Correct']
headers = list(readme_table.columns)
lines = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
for values in readme_table.astype(str).itertuples(index=False, name=None):
    lines.append('| ' + ' | '.join(value.replace('|', '/') for value in values) + ' |')
table_text = '\n'.join(lines)
summary = f'Local 2026 World Cup audit through 2026-07-03: {audit.correct.eq("Yes").sum()}/{len(audit)} correct ({accuracy:.2%}).'
(REPORTS / 'world_cup_2026_readme_table.txt').write_text(summary + '\n\n' + table_text + '\n', encoding='utf-8')

assert len(audit) == 88
assert audit.correct.isin(['Yes', 'No']).all()
print(summary)
audit.head()


Local 2026 World Cup audit through 2026-07-03: 55/88 correct (62.50%).


,date,match,score,model_prediction,actual_result,correct,home_win_probability,draw_probability,away_win_probability
0,2026-06-11,Mexico vs South Africa,2-0,Mexico,Mexico,Yes,0.823793,0.127302,0.048906
1,2026-06-11,South Korea vs Czech Republic,2-1,South Korea,South Korea,Yes,0.475353,0.220242,0.304404
2,2026-06-12,Canada vs Bosnia and Herzegovina,1-1,Canada,Draw,No,0.709085,0.171329,0.119585
3,2026-06-12,United States vs Paraguay,4-1,Paraguay,United States,No,0.321360,0.217650,0.460990
4,2026-06-13,Qatar vs Switzerland,1-1,Switzerland,Draw,No,0.051292,0.103379,0.845329
